# Indian Invoice & Bill Extraction Pipeline - YOLOv8 Training (Google Colab)

> [!WARNING]
> **AUDITING NOTE**: Cells 4-7 and 9-10 are superseded. Known errors in cell 10:
> - Hard-coded baseline values 0.352/0.391/0.337 are wrong (measured 0.337/0.391/0.353).
> - The rule silently changed to 3 pairs.
> - Baseline seeds 3-4 are missing.
> - The weights path assumes runs/detect/<name> while the real path is runs/detect/runs/detect/<name>.

Self-contained notebook for training and validating the YOLOv8 field detector on Google Colab with GPU acceleration (NVIDIA T4).

### Workflow:
1. Mount Google Drive
2. Extract self-contained `code.zip` and `yolo_v1.zip` from Drive folder `invoice-vision` into `/content/invoice-vision`
3. Set `PYTHONPATH=src` and install `ultralytics==8.4.215` only (preserves Colab preinstalled CUDA/Torch/OpenCV environment)
4. Inspect GPU hardware via `nvidia-smi`
5. Prepare real-only dataset copy by removing auto-transferred synthetic photo variations (`_mild_`, `_medium_`, `_severe_`, `_hard_`)
6. Train baseline models across 3 seeds (`seed=0, 1, 2`) with run names `baseline_s0`, `baseline_s1`, `baseline_s2`
7. Evaluate on **VALIDATION ONLY** after each run, syncing weights and logs to Drive
8. Multi-seed Validation Summary: per-class mean and min-max of AP50 and Recall
9. Final Cell (Frozen Config): Run ONCE on **TEST** set

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Extract Codebase and Dataset

Unpacks `invoice-vision/code.zip` and `invoice-vision/yolo_v1.zip` into `/content/invoice-vision`.

In [ ]:
import os
from pathlib import Path

WORK_DIR = Path("/content/invoice-vision")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(str(WORK_DIR))

CODE_ZIP = Path("/content/drive/MyDrive/invoice-vision/code.zip")
DATA_ZIP = Path("/content/drive/MyDrive/invoice-vision/yolo_v1.zip")

assert CODE_ZIP.exists(), f"Missing {CODE_ZIP}. Please upload code.zip to Drive under 'invoice-vision/'."
assert DATA_ZIP.exists(), f"Missing {DATA_ZIP}. Please upload yolo_v1.zip to Drive under 'invoice-vision/'."

# Unpack code (configs/, scripts/, src/, requirements.txt)
!unzip -q -o /content/drive/MyDrive/invoice-vision/code.zip -d /content/invoice-vision

# Unpack dataset (images/, labels/, data.yaml)
!mkdir -p /content/invoice-vision/data/yolo_v1
!unzip -q -o /content/drive/MyDrive/invoice-vision/yolo_v1.zip -d /content/invoice-vision/data/yolo_v1

print("Code and dataset extracted successfully:")
!ls -la /content/invoice-vision
!ls -la /content/invoice-vision/data/yolo_v1

## 3. Environment Setup & Dependency Installation

Installs `ultralytics==8.4.215` only. Never run `pip install -r requirements.txt` on Colab to prevent breaking preinstalled PyTorch, TorchVision, and CUDA runtime.

In [ ]:
import os
os.environ["PYTHONPATH"] = "src"

# Install pinned ultralytics version only - do not alter preinstalled Colab torch/cuda/numpy
!pip install -q ultralytics==8.4.215

import torch
import ultralytics
print(f"Ultralytics Version : {ultralytics.__version__}")
print(f"PyTorch Version     : {torch.__version__}")
print(f"CUDA Available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")

## 4. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 5. Prepare Real-Only Dataset (Delete Auto-Transferred Synthetic Photos)

Per Experiment 1 findings in `docs/experiments.md`, auto-transferred synthetic photos from the 4 seed invoices do not improve generalization. Deletes training files containing `_mild_`, `_medium_`, `_severe_`, or `_hard_` to maintain a strict real-only baseline ($n=33$ real training images).

In [ ]:
from pathlib import Path

train_img_dir = Path("/content/invoice-vision/data/yolo_v1/images/train")
train_lbl_dir = Path("/content/invoice-vision/data/yolo_v1/labels/train")

synth_keywords = ["_mild_", "_medium_", "_severe_", "_hard_"]
deleted_images = 0
deleted_labels = 0

for p in list(train_img_dir.glob("*")):
    if any(k in p.name for k in synth_keywords):
        p.unlink()
        deleted_images += 1
        lbl_p = train_lbl_dir / f"{p.stem}.txt"
        if lbl_p.exists():
            lbl_p.unlink()
            deleted_labels += 1

remaining_train = len(list(train_img_dir.glob("*.jpg"))) + len(list(train_img_dir.glob("*.png")))
print(f"Removed {deleted_images} synthetic images and {deleted_labels} labels.")
print(f"Remaining Real-Only Training Images: {remaining_train} (expected 33)")

## 6. Multi-Seed Baseline Training (VAL Evaluation Only)

Trains the baseline architecture across 3 seeds (`0, 1, 2`) with run names `baseline_s0`, `baseline_s1`, `baseline_s2`.
After each run:
- Evaluates strictly on **VAL** split (`--splits val`).
- Synchronizes model weights, confusion matrices, and markdown reports to Google Drive under `invoice-vision/runs/<run-name>`.

In [ ]:
seeds = [0, 1, 2]

for seed in seeds:
    run_name = f"baseline_s{seed}"
    print(f"\n{'='*70}")
    print(f"STARTING BASELINE RUN: {run_name} (Seed: {seed})")
    print(f"{'='*70}")
    
    # 1. Train detector
    !python scripts/train_yolo.py \
        --config configs/yolo_v1.yaml \
        --data-root data/yolo_v1 \
        --seed {seed} \
        --run-name {run_name}
    
    # 2. Evaluate on VAL split only
    best_weights = f"runs/detect/{run_name}/weights/best.pt"
    val_out_md = f"results/val_eval_{run_name}.md"
    
    !python scripts/eval_yolo.py \
        --weights {best_weights} \
        --data-root data/yolo_v1 \
        --splits val \
        --imgsz 1024 \
        --out {val_out_md} \
        --overlays-dir results/overlays
    
    # 3. Synchronize run outputs and report to Google Drive
    drive_dest = f"/content/drive/MyDrive/invoice-vision/runs/{run_name}"
    !mkdir -p {drive_dest}
    !cp -r runs/detect/{run_name}/* {drive_dest}/
    !cp {val_out_md} {drive_dest}/
    print(f"Completed {run_name}. Artifacts backed up to {drive_dest}.")

## 7. Multi-Seed Validation Summary & Comparison

Aggregates validation performance across seeds 0, 1, and 2, reporting the **mean and min-max** of AP50 and Recall for each class on the validation set.

In [ ]:
import re
import pandas as pd
from pathlib import Path

classes = ["vendor", "invoice_no", "date", "gstin", "table", "total"]
metrics = {c: {"ap50": [], "recall": []} for c in classes}

for seed in [0, 1, 2]:
    report_path = Path(f"results/val_eval_baseline_s{seed}.md")
    if not report_path.exists():
        print(f"Warning: {report_path} not found. Skipping.")
        continue
    
    txt = report_path.read_text(encoding="utf-8")
    for line in txt.splitlines():
        for c in classes:
            if f"`{c}`" in line:
                parts = [p.strip() for p in line.split("|") if p.strip()]
                if len(parts) >= 7:
                    try:
                        rec = float(parts[4])
                        ap50 = float(parts[5])
                        metrics[c]["recall"].append(rec)
                        metrics[c]["ap50"].append(ap50)
                    except ValueError:
                        pass

summary_rows = []
for c in classes:
    ap50_vals = metrics[c]["ap50"]
    rec_vals = metrics[c]["recall"]
    if ap50_vals and rec_vals:
        summary_rows.append({
            "Class": c,
            "Mean AP50": f"{sum(ap50_vals)/len(ap50_vals):.4f}",
            "AP50 (Min - Max)": f"{min(ap50_vals):.4f} - {max(ap50_vals):.4f}",
            "Mean Recall": f"{sum(rec_vals)/len(rec_vals):.4f}",
            "Recall (Min - Max)": f"{min(rec_vals):.4f} - {max(rec_vals):.4f}",
            "Sample Size Viability": "[TOO FEW (< 15 in Val)]",
        })

df_summary = pd.DataFrame(summary_rows)
print("\n" + "="*75)
print("VALIDATION METRIC SUMMARY ACROSS 3 SEEDS (seeds 0, 1, 2)")
print("="*75)
print(df_summary.to_string(index=False))

# Save summary to Drive
summary_csv = Path("/content/drive/MyDrive/invoice-vision/runs/val_multi_seed_summary.csv")
df_summary.to_csv(summary_csv, index=False)
print(f"\nSaved multi-seed validation summary to: {summary_csv}")

## 8. Experiment 2: Extract Multi-Layout Synthetic Dataset (`yolo_v2`)

Extracts `invoice-vision/yolo_v2.zip` containing 57 train images (33 real + 24 synthetic multi-layout), 12 val, and 14 test images. Verifies counts and split integrity prior to training.

In [ ]:
import os
from pathlib import Path

YOLO_V2_ZIP = Path("/content/drive/MyDrive/invoice-vision/yolo_v2.zip")
assert YOLO_V2_ZIP.exists(), f"Missing {YOLO_V2_ZIP}. Please upload yolo_v2.zip to Drive under 'invoice-vision/'."

# Unpack dataset into /content/invoice-vision/data/yolo_v2
!mkdir -p /content/invoice-vision/data/yolo_v2
!unzip -q -o /content/drive/MyDrive/invoice-vision/yolo_v2.zip -d /content/invoice-vision/data/yolo_v2

v2_train = len(list(Path("/content/invoice-vision/data/yolo_v2/images/train").glob("*")))
v2_val = len(list(Path("/content/invoice-vision/data/yolo_v2/images/val").glob("*")))
v2_test = len(list(Path("/content/invoice-vision/data/yolo_v2/images/test").glob("*")))

print("yolo_v2 dataset extracted successfully:")
print(f"  Train : {v2_train} images (expected 57: 33 real + 24 synthetic multi-layout)")
print(f"  Val   : {v2_val} images (expected 12, identical to yolo_v1)")
print(f"  Test  : {v2_test} images (expected 14, identical to yolo_v1)")

assert v2_train == 57, f"Expected 57 train images, found {v2_train}"
assert v2_val == 12, f"Expected 12 val images, found {v2_val}"
assert v2_test == 14, f"Expected 14 test images, found {v2_test}"

## 9. Experiment 2: Multi-Seed Training (Seeds 0 – 4, VAL Evaluation Only)

Trains YOLOv8n across 5 independent seeds (`seed=0, 1, 2, 3, 4`) on `data/yolo_v2`.
After each run:
- Evaluates strictly on **VAL** split (`--splits val`).
- Synchronizes model weights, validation predictions, and reports to Google Drive under `invoice-vision/runs/<run-name>`.

In [ ]:
exp2_seeds = [0, 1, 2, 3, 4]

for seed in exp2_seeds:
    run_name = f"yolo_v2_s{seed}"
    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT 2 RUN: {run_name} (Seed: {seed})")
    print(f"{'='*70}")
    
    # 1. Train detector on yolo_v2
    !python scripts/train_yolo.py \
        --config configs/yolo_v1.yaml \
        --data-root data/yolo_v2 \
        --seed {seed} \
        --run-name {run_name}
    
    # 2. Evaluate strictly on VAL split
    best_weights = f"runs/detect/{run_name}/weights/best.pt"
    val_out_md = f"results/val_eval_{run_name}.md"
    
    !python scripts/eval_yolo.py \
        --weights {best_weights} \
        --data-root data/yolo_v2 \
        --splits val \
        --imgsz 1024 \
        --out {val_out_md} \
        --overlays-dir results/overlays
    
    # 3. Synchronize run outputs and report to Google Drive
    drive_dest = f"/content/drive/MyDrive/invoice-vision/runs/{run_name}"
    !mkdir -p {drive_dest}
    !cp -r runs/detect/{run_name}/* {drive_dest}/
    !cp {val_out_md} {drive_dest}/
    print(f"Completed {run_name}. Artifacts backed up to {drive_dest}.")

## 10. Experiment 2: Validation Evaluation & Pre-Registered Decision Rule

Evaluates validation performance against the pre-registered decision rule:
1. **Mean Gain Requirement**: $\Delta \text{mAP50} \ge +0.05$ over real baseline ($0.360 \to \ge 0.410$).
2. **Consistency Requirement**: At least 4 of 5 same-seed pairs favor `data/yolo_v2`.
3. If both pass: confirm empirical improvement. Otherwise: report "no evidence of improvement".

In [ ]:
import re
import pandas as pd
from pathlib import Path

# Baseline validation mAP50 for seeds 0, 1, 2 from Experiment 1 (docs/experiments.md)
baseline_map50 = {
    0: 0.352,
    1: 0.391,
    2: 0.337,
}
baseline_mean_map50 = 0.360

v2_map50 = {}
classes = ["vendor", "invoice_no", "date", "gstin", "table", "total"]
class_metrics = {c: [] for c in classes}

for seed in [0, 1, 2, 3, 4]:
    report_path = Path(f"results/val_eval_yolo_v2_s{seed}.md")
    if not report_path.exists():
        print(f"Notice: {report_path} not found. Skipping.")
        continue
    
    txt = report_path.read_text(encoding="utf-8")
    for line in txt.splitlines():
        if "Overall mAP@0.50" in line or "| **all** |" in line or "mAP50" in line:
            m = re.search(r"(\d+\.\d+)", line)
            if m:
                v2_map50[seed] = float(m.group(1))
                break

print("\n" + "="*75)
print("EXPERIMENT 2: VALIDATION DECISION RULE EVALUATION")
print("="*75)
if v2_map50:
    v2_mean = sum(v2_map50.values()) / len(v2_map50)
    gain = v2_mean - baseline_mean_map50
    
    pairs_favoring = sum(1 for s in v2_map50 if s in baseline_map50 and v2_map50[s] > baseline_map50[s])
    total_pairs = sum(1 for s in v2_map50 if s in baseline_map50)
    
    print(f"Baseline Mean mAP50 (seeds 0-2)  : {baseline_mean_map50:.4f}")
    print(f"yolo_v2 Mean mAP50 ({len(v2_map50)} seeds)      : {v2_mean:.4f} (gain: {gain:+.4f})")
    print(f"Same-seed pairs favoring yolo_v2  : {pairs_favoring}/{total_pairs}")
    print("-" * 75)
    
    gain_pass = gain >= 0.05
    pairs_pass = pairs_favoring >= 4 if total_pairs >= 4 else (pairs_favoring == total_pairs)
    
    if gain_pass and pairs_pass:
        print("[DECISION RESULT]: EMPIRICAL IMPROVEMENT CONFIRMED (Gain >= +0.05 and >= 4/5 seed pairs).")
        print("Decision: Adopt data/yolo_v2 as the new production training baseline.")
    else:
        print("[DECISION RESULT]: NO EVIDENCE OF IMPROVEMENT.")
        print("Decision: Retain the 33-image real-only baseline per pre-registered decision rule.")
else:
    print("No yolo_v2 validation runs found yet.")
print("="*75)

## 11. Experiment 3: Extract Expanded Multi-Layout Synthetic Dataset (`yolo_v3`)

Extracts `invoice-vision/yolo_v3.zip` containing 117 train images (33 real + 84 synthetic multi-layout), 12 val, and 14 test images. Verifies counts and split integrity prior to training.

In [ ]:
import os
from pathlib import Path

YOLO_V3_ZIP = Path("/content/drive/MyDrive/invoice-vision/yolo_v3.zip")
assert YOLO_V3_ZIP.exists(), f"Missing {YOLO_V3_ZIP}. Please upload yolo_v3.zip to Drive under 'invoice-vision/'."

# Unpack dataset into /content/invoice-vision/data/yolo_v3
!mkdir -p /content/invoice-vision/data/yolo_v3
!unzip -q -o /content/drive/MyDrive/invoice-vision/yolo_v3.zip -d /content/invoice-vision/data/yolo_v3

v3_train = len(list(Path("/content/invoice-vision/data/yolo_v3/images/train").glob("*")))
v3_val = len(list(Path("/content/invoice-vision/data/yolo_v3/images/val").glob("*")))
v3_test = len(list(Path("/content/invoice-vision/data/yolo_v3/images/test").glob("*")))

print("yolo_v3 dataset extracted successfully:")
print(f"  Train : {v3_train} images (expected 117: 33 real + 84 synthetic multi-layout)")
print(f"  Val   : {v3_val} images (expected 12, identical to yolo_v2)")
print(f"  Test  : {v3_test} images (expected 14, identical to yolo_v2)")

assert v3_train == 117, f"Expected 117 train images, found {v3_train}"
assert v3_val == 12, f"Expected 12 val images, found {v3_val}"
assert v3_test == 14, f"Expected 14 test images, found {v3_test}"

## 12. Experiment 3: Multi-Seed Training (Seeds 0 – 4, VAL Evaluation Only)

Trains YOLOv8n across 5 independent seeds (`seed=0, 1, 2, 3, 4`) on `data/yolo_v3` with run names `v3_s0` to `v3_s4`.
After each run:
- Evaluates strictly on **VAL** split (`--splits val`).
- Synchronizes model weights, validation predictions, and reports to Google Drive under `invoice-vision/runs/<run-name>`.

In [ ]:
from pathlib import Path

exp3_seeds = [0, 1, 2, 3, 4]

for seed in exp3_seeds:
    run_name = f"v3_s{seed}"
    print(f"\n{'='*70}")
    print(f"STARTING EXPERIMENT 3 RUN: {run_name} (Seed: {seed})")
    print(f"{'='*70}")
    
    # 1. Train detector on yolo_v3
    !python scripts/train_yolo.py \
        --config configs/yolo_v1.yaml \
        --data-root data/yolo_v3 \
        --seed {seed} \
        --run-name {run_name}
    
    # 2. Locate weights (supporting nested Ultralytics path)
    cand_nested = Path(f"runs/detect/runs/detect/{run_name}/weights/best.pt")
    cand_flat = Path(f"runs/detect/{run_name}/weights/best.pt")
    best_weights = str(cand_nested if cand_nested.exists() else cand_flat)
    run_dir = cand_nested.parent.parent if cand_nested.exists() else cand_flat.parent.parent
    
    # 3. Evaluate strictly on VAL split
    val_out_md = f"results/val_eval_{run_name}.md"
    
    !python scripts/eval_yolo.py \
        --weights {best_weights} \
        --data-root data/yolo_v3 \
        --splits val \
        --imgsz 1024 \
        --out {val_out_md} \
        --overlays-dir results/overlays
    
    # 4. Synchronize run outputs and report to Google Drive
    drive_dest = f"/content/drive/MyDrive/invoice-vision/runs/{run_name}"
    !mkdir -p {drive_dest}
    !cp -r {run_dir}/* {drive_dest}/
    !cp {val_out_md} {drive_dest}/
    print(f"Completed {run_name}. Artifacts backed up to {drive_dest}.")


## 13. Experiment 3: Validation Evaluation & Pre-Registered Decision Rule (v3 vs. Layouts Baseline)

> **Changelog (2026-11-22)**: The earlier text said >= 0.05, which differed from the rule specified in the original instructions; corrected to mean gain >= +0.03 (target >= 0.476) and >= 4/5 seed pairs before any Experiment 3 result existed.

Evaluates validation performance against the pre-registered decision rule:
1. **Mean Gain Requirement**: $\Delta \text{mAP50} \ge +0.03$ over the measured yolo_v2 mean 0.446 (target $\ge 0.476$).
2. **Consistency Requirement**: At least 4 of 5 same-seed pairs favor `data/yolo_v3` over `data/yolo_v2`.
3. If both pass: confirm empirical improvement. Otherwise: report "no evidence of improvement".

In [ ]:
import re
import pandas as pd
from pathlib import Path

# Measured validation mAP50 for seeds 0 - 4 from Experiment 2 (docs/experiments.md)
# Layouts (data/yolo_v2, 57 images) baseline values:
layouts_map50 = {
    0: 0.514,
    1: 0.460,
    2: 0.408,
    3: 0.416,
    4: 0.430,
}
layouts_mean_map50 = 0.446

v3_map50 = {}
classes = ["vendor", "invoice_no", "date", "gstin", "table", "total"]

for seed in [0, 1, 2, 3, 4]:
    run_name = f"v3_s{seed}"
    report_path = Path(f"results/val_eval_{run_name}.md")
    if not report_path.exists():
        print(f"Notice: {report_path} not found. Skipping.")
        continue
    
    txt = report_path.read_text(encoding="utf-8")
    for line in txt.splitlines():
        if "Overall mAP@0.50" in line or "| **all** |" in line or "mAP50" in line:
            m = re.search(r"(\d+\.\d+)", line)
            if m:
                v3_map50[seed] = float(m.group(1))
                break

print("\n" + "="*75)
print("EXPERIMENT 3: VALIDATION DECISION RULE EVALUATION (v3 vs. Layouts)")
print("="*75)
if v3_map50:
    v3_mean = sum(v3_map50.values()) / len(v3_map50)
    gain = v3_mean - layouts_mean_map50
    
    pairs_favoring = sum(1 for s in v3_map50 if s in layouts_map50 and v3_map50[s] > layouts_map50[s])
    total_pairs = sum(1 for s in v3_map50 if s in layouts_map50)
    
    print(f"Layouts Baseline Mean mAP50 (seeds 0-4) : {layouts_mean_map50:.4f}")
    print(f"yolo_v3 Mean mAP50 ({len(v3_map50)} seeds)              : {v3_mean:.4f} (gain: {gain:+.4f})")
    print(f"Same-seed pairs favoring yolo_v3        : {pairs_favoring}/{total_pairs}")
    print("-" * 75)
    
    gain_pass = gain >= 0.03
    pairs_pass = pairs_favoring >= 4 if total_pairs >= 4 else (pairs_favoring == total_pairs)
    
    if gain_pass and pairs_pass:
        print("[DECISION RESULT]: EMPIRICAL IMPROVEMENT CONFIRMED (Gain >= +0.03 and >= 4/5 seed pairs, target >= 0.476).")
        print("Decision: Adopt data/yolo_v3 as the new production training baseline.")
    else:
        print("[DECISION RESULT]: NO EVIDENCE OF IMPROVEMENT.")
        print("Decision: Retain data/yolo_v2 (layouts) baseline per pre-registered decision rule.")
else:
    print("No v3 validation runs found yet.")
print("="*75)


## 14. RUN ONCE, after the config is frozen (Test Set Evaluation)

> **CRITICAL AUDITING RULE**: Execute this cell **ONCE ONLY** after model architecture, hyperparameters, and augmentations are fully frozen based on validation metrics. Never touch the test set during tuning or exploration.

In [ ]:
# RUN ONCE, after the config is frozen
# Select the best performing seed checkpoint from validation (e.g., baseline_s0)
CHOSEN_RUN = "baseline_s0"
chosen_weights = f"runs/detect/{CHOSEN_RUN}/weights/best.pt"
test_out_md = f"results/test_eval_final_{CHOSEN_RUN}.md"

print(f"Executing single final TEST evaluation using: {chosen_weights}")
!python scripts/eval_yolo.py \
    --weights {chosen_weights} \
    --data-root data/yolo_v1 \
    --splits test \
    --imgsz 1024 \
    --out {test_out_md} \
    --overlays-dir results/overlays

# Back up final test evaluation to Google Drive
!cp {test_out_md} /content/drive/MyDrive/invoice-vision/
!cp -r results/overlays/test /content/drive/MyDrive/invoice-vision/results/overlays/
print(f"Final test evaluation saved to {test_out_md} and copied to Google Drive!")